# 6. Interactive Map with Folium
Mark launch sites, success/failure launches, and distances to nearby places. Screenshot each map for the report.

In [14]:
%pip install folium

Note: you may need to restart the kernel to use updated packages.


In [15]:
import os
import folium
import pandas as pd
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon
from math import sin, cos, sqrt, atan2, radians

url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
local = '../data/spacex_launch_geo.csv'

try:
    spacex_df = pd.read_csv(url)
    spacex_df.to_csv(local, index=False)      # keep a local copy for later
    print('Loaded from the web and saved a local copy')
except Exception as e:
    print('Web load failed:', type(e).__name__)
    if os.path.exists(local):
        spacex_df = pd.read_csv(local)
        print('Loaded the local copy instead')
    else:
        raise SystemExit('No internet and no local file. Download the CSV in your browser, '
                         'save it as ../data/spacex_launch_geo.csv, then run this cell again.')

spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby('Launch Site', as_index=False).first()[['Launch Site', 'Lat', 'Long']]
launch_sites_df

Loaded from the web and saved a local copy


,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


In [16]:
# Task 1: mark all launch sites
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
for _, row in launch_sites_df.iterrows():
    coord = [row['Lat'], row['Long']]
    folium.Circle(coord, radius=1000, color='#d35400', fill=True).add_child(folium.Popup(row['Launch Site'])).add_to(site_map)
    folium.map.Marker(coord, icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
        html='<div style="font-size:12; color:#d35400;"><b>%s</b></div>' % row['Launch Site'])).add_to(site_map)
site_map

In [17]:
# Task 2: success (green) / failure (red) launch markers per site
spacex_df['marker_color'] = spacex_df['class'].apply(lambda c: 'green' if c == 1 else 'red')
marker_cluster = MarkerCluster()
site_map.add_child(marker_cluster)
for _, row in spacex_df.iterrows():
    marker_cluster.add_child(folium.Marker([row['Lat'], row['Long']],
        icon=folium.Icon(color='white', icon_color=row['marker_color']),
        popup=row['Launch Site']))
site_map

In [18]:
# Task 3: distance from a launch site to a nearby feature
MousePosition().add_to(site_map)   # hover over the map to read coordinates

def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    a = sin((lat2 - lat1) / 2) ** 2 + cos(lat1) * cos(lat2) * sin((lon2 - lon1) / 2) ** 2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

# EXAMPLE coordinates: replace with the points you read off the map yourself
launch_site_lat, launch_site_lon = 28.56367, -80.57163   # CCAFS SLC-40
coast_lat, coast_lon = 28.56342, -80.56794               # nearest coastline point (example)
distance_coast = calculate_distance(launch_site_lat, launch_site_lon, coast_lat, coast_lon)
print('Distance to coast: %.2f km' % distance_coast)

folium.Marker([coast_lat, coast_lon], icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
    html='<div style="font-size:12; color:#252526;"><b>%.2f KM</b></div>' % distance_coast)).add_to(site_map)
folium.PolyLine([[launch_site_lat, launch_site_lon], [coast_lat, coast_lon]], weight=1).add_to(site_map)
site_map

Distance to coast: 0.36 km


Repeat the last cell for the nearest **railway**, **highway** and **city** (read their coordinates from the map) and note what the distances suggest about how launch sites are chosen.